In [4]:
import torch 
from torch import nn, Tensor 
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from typing import override 
import matplotlib.pyplot as plt 
from enum import Enum
import numpy as np 
import re 

import ipywidgets as widgets
from IPython.display import display
from pprint import pprint

class Mode(Enum):
    CNN = 1
    BasicTransformer = 2

    def to_string(self):
        return self.name

MODE = Mode.BasicTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"


# ================================================================
# Version 1: CNN Architecture
# ================================================================
class FlowMatching(nn.Module):
    def __init__(self) -> None:
        super().__init__()

        self.digit_embedding = nn.Embedding(10, 64)
        self.time_embedding = nn.Sequential(
            nn.Linear(1, 64),
            nn.SiLU(),
            nn.Linear(64, 64)
        )
        self.net = nn.Sequential(
            nn.Conv2d(1 + 64 + 64, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(128, 128, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(128, 64, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(64, 1, 3, padding=1),
        )



    @override 
    def forward(self, x, t, digit):

        B, _, H, W = x.shape 
        
        digit_emb = self.digit_embedding(digit)

        # Embed digit: [B, 64] -> [B, 64, H, W]
        digit_emb = digit_emb[:, :, None, None]
        digit_emb = digit_emb.expand(-1, -1, H, W)

        # Embed time: [B] -> [B, 1] -> [B, 64]
        time_emb = self.time_embedding(t[:, None])

        # [B, 64] -> [B, 64, H, W]
        time_emb = time_emb[:, :, None, None].expand(-1, -1, H, W)

        # concatenate everything 
        x = torch.cat([x, digit_emb, time_emb], dim=1)

        return self.net(x)


# ================================================================
# version 2: transformer architecture
# ================================================================
class FlowMatching_with_Transformer(nn.Module):
    def __init__(self):
        super().__init__() 

        self.embedding_dim=64

        self.digit_embedding = nn.Embedding(
            num_embeddings=10,
            embedding_dim=self.embedding_dim
        )

        self.time_embedding = nn.Sequential(
            nn.Linear(1, self.embedding_dim),
            nn.SiLU(),
            nn.Linear(self.embedding_dim, self.embedding_dim),
        )

        self.patch_embedding = nn.Conv2d(
            in_channels=1,
            out_channels=self.embedding_dim,
            kernel_size=4,
            stride=4,
        )

        self.pos_embedding = nn.Parameter(
            torch.randn(1, 49, self.embedding_dim)
        )

        self.num_layers = 4
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=self.embedding_dim,
                nhead=4,
                dim_feedforward=128,
                batch_first=True
            )
            for _ in range(self.num_layers)
        ])

        self.output_proj = nn.Linear(
            self.embedding_dim, 
            4 * 4,
        )



    @override
    def forward(self, x, t, digit):

        digit_emb = self.digit_embedding(digit)
        digit_emb = digit_emb.unsqueeze(1)

        time_emb = self.time_embedding(t.unsqueeze(1)).unsqueeze(1)

        patch = self.patch_embedding(x)
        patch = patch.flatten(2)
        patch = patch.transpose(1, 2)

        image_tokens = patch + self.pos_embedding

        x = torch.cat([
            digit_emb, 
            time_emb,
            image_tokens,
            ], dim=1
        )

        for layer in self.layers:
            x = layer(x)


        # remove digit and time tokens again 
        image_tokens = x[:, 2:, :]

        # convert each token back into a 4x4 patch 
        patches = self.output_proj(image_tokens)
        patches = patches.transpose(1, 2) # back to [B, 16, 49]

        # reconstruct the velocity image 
        velocity = F.fold(
            patches, 
            output_size=(28, 28),
            kernel_size=4,
            stride=4,
        ) # [B, 1, 28, 28]

        return velocity

# ================================================================
# Training
# ================================================================

def create_loader(training_size=1024, batch_size=128) -> DataLoader:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    dataset = datasets.MNIST(
        root="./data", 
        train=True,
        download=True,
        transform=transform,
    )

    indices = torch.randperm(len(dataset))[:training_size]
    dataset = Subset(dataset, indices)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
    )


def train(model, loader, training_epochs=20):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    for epoch in range(training_epochs):

        total_loss = 0 

        for x1, digit in loader:

            x1 = x1.to(device)
            digit = digit.to(device)

            B = x1.shape[0]
            x0 = torch.randn_like(x1) # gausssian source distribution

            t = torch.rand(B, device=device)

            t_img = t[:, None, None, None]
            xt = ((1 - t_img) * x0 + t_img * x1)

            target_velocity = x1 - x0
            predicted_velocity = model(xt, t, digit)

            loss = F.mse_loss(
                predicted_velocity,
                target_velocity,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step() 

            total_loss += loss.item() 

        print(
            f"Epoch {epoch + 1}: "
            f"loss = {total_loss / len(loader):.4f}"
        )

@torch.no_grad()
def generate(model, digits, steps=10):

    model.eval() 

    B = len(digits)
    x = torch.randn(
        B, 1, 28, 28,
        device=device,
    )

    dt = 1.0 / steps

    for i in range(steps):
        t = torch.full(
            (B,),
            i / steps,
            device=device
        ) # Creates tensor of batch_size with the current time step

        velocity = model(
            x,
            t,
            digits,
        )

        x += dt * velocity

    return x 

def show_all_digits(model):
    digits = torch.arange(10, device=device)
    samples = generate(
        model, digits, steps=50
    )

    fig, axes = plt.subplots(1, 10, figsize=(15, 2))

    for i, ax in enumerate(axes):
        ax.imshow(
            samples[i, 0].cpu(),
            cmap="gray",
        )

        ax.set_title(str(i))
        ax.axis("off")
    plt.show()

def save_model(model) -> None:
    filename = f"{MODE.to_string()}.pt"
    model.to("cpu")
    model.eval()
    torch.save(model.state_dict(), filename)

def load_model(model):
    filename = f"{MODE.to_string()}.pt"
    model.load_state_dict(torch.load(filename, map_location=device))
    return model

def visualize_params(state_dict, initial_state_dict=None):
    """
    Interactive visualization of parameter statistics across Transformer layers.

    Parameters 
    ---------- 
    model: 
        Traned/current PyTorch model. 

    initial_state_dict: 
        State dict of the model before training.
        If provided, additional metrics such as relative parameter change become 
        available. 

    Example usage before training: 

    inital_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }
    """

    layer_params = {}

    pattern = re.compile(
        r"^layers\.(\d+)\.(.+)$"
    )

    for name, tensor in state_dict:

        match = pattern.match(name)

        if match is None:
            continue

        layer_idx = int(match.group(1))
        param_name = match.group(2)

        # Only parameters, not buffers
        if not torch.is_tensor(tensor):
            continue

        layer_params.setdefault(layer_idx, {})
        layer_params[layer_idx][param_name] = tensor.detach().cpu().float()

    if not layer_params: 
        raise ValueError(
            "Could not automatically find Transformer layers. "
            "Check model.named_parameters()."
        )

    def categorize(name):
        if "self_attn.in_proj" in name:
            return "Attention QKV"
        if "self_attn.out_proj" in name: 
            return "Attention Output"
        if "linear1" in name: 
            return "FFN Input"
        if "linear2" in name: 
            return "FFN Output"
        if "norm1" in name: 
            return "LayerNorm 1"
        if "norm2" in name: 
            return "LayerNorm 2"
    
        return "Other"

    categories = [
        "Attention QKV",
        "Attention Output",
        "FFN Input", 
        "FFN Output",
        "LayerNorm 1",
        "LayerNorm 2",
        "Other",
    ]
    category_selector = widgets.SelectMultiple(
        options=categories,
        value=tuple(categories[:4]),
        description="Params:",
        layout=widgets.Layout(
            width="300px",
            height="150px"
        ),
    )

    metric_options = [
        "L2 Norm",
        "Mean absolute value",
        "Standard deviation",
        "Relative change",
        "Absolute change",
    ]

    if initial_state_dict is None:
        metric_options = metric_options[:3]

    metric_selector = widgets.Dropdown(
        options=metric_options,
        value=metric_options[0],
        description="Metric:",
        layout=widgets.Layout(width="300px"),
    )

    log_scale = widgets.Checkbox(
        value=False,
        description="Log scale"

    )

    output = widgets.Output()

    # Calculate metrics 

    def calculate_metric(full_name, tensor, metric):

        tensor = tensor.detach().cpu().float()

        if metric == "L2 norm":
            return float(torch.linalg.vector_norm(tensor))
        elif metric == "Mean absolute value":
            return float(tensor.abs().mean())
        elif metric == "Standard deviation":
            return float(tensor.std())

        # Metrics below require initial_state_dict
        if initial_state_dict is None:
            return np.nan
        if full_name not in initial_state_dict:
            return np.nan

        initial = (
            initial_state_dict[full_name]
            .detach()
            .cpu()
            .float()
        )

        difference = tensor - initial

        if metric == "Absolute change":
            return float(
                torch.linalg.vector_norm(difference)
            )

        elif metric == "Relative change":

            epsilon = 1e-8

            return float(
                torch.linalg.vector_norm(difference)
                /
                (
                    torch.linalg.vector_norm(initial)
                    + epsilon
                )
            )
        return np.nan

    def update(*args):

        with output:
            output.clear_output(wait=True)

            selected = category_selector.value
            metric = metric_selector.value

            if not selected:
                print("Select at least one parameter group.")
                return

            matrix = []
            row_labels = []

            layer_indices = sorted(layer_params.keys())

            for selected_category in selected:
                values = []

                for layer_idx in layer_indices:

                    category_values = []

                    for param_name, tensor in layer_params[layer_idx].items():

                        # Check whether this parameter belongs
                        # to the selected category
                        if categorize(param_name) != selected_category:
                            continue

                        full_name = (
                            f"layers.{layer_idx}.{param_name}"
                        )

                        value = calculate_metric(
                            full_name,
                            tensor,
                            metric,
                        )

                        # Only add valid values
                        if value is not None and np.isfinite(value):
                            category_values.append(float(value))

                    # --------------------------------------------
                    # Important: handle missing parameters
                    # --------------------------------------------

                    if category_values:
                        values.append(
                            float(np.mean(category_values))
                        )
                    else:
                        values.append(np.nan)

                matrix.append(values)
                row_labels.append(selected_category)

            matrix = np.asarray(matrix, dtype=float)

            # --------------------------------------------
            # Plot
            # --------------------------------------------

            plot_matrix = matrix.copy()

            if log_scale.value:
                plot_matrix = np.log10(
                    np.maximum(
                        np.abs(plot_matrix),
                        1e-12,
                    )
                )

            fig, ax = plt.subplots(
                figsize=(
                    max(8, len(layer_indices) * 1.3),
                    max(4, len(row_labels) * 0.8),
                )
            )

            im = ax.imshow(
                plot_matrix,
                aspect="auto",
                interpolation="nearest",
            )

            ax.set_xticks(range(len(layer_indices)))
            ax.set_xticklabels(
                [f"Layer {i}" for i in layer_indices]
            )

            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels)

            ax.set_xlabel("Transformer layer")
            ax.set_ylabel("Parameter group")

            ax.set_title(
                f"Transformer parameters — {metric}"
            )

            plt.colorbar(
                im,
                ax=ax,
                label=(
                    f"log10({metric})"
                    if log_scale.value
                    else metric
                ),
            )

            for row in range(matrix.shape[0]):
                for col in range(matrix.shape[1]):

                    value = matrix[row, col]

                    if np.isfinite(value):
                        ax.text(
                            col,
                            row,
                            f"{value:.3g}",
                            ha="center",
                            va="center",
                            color="white",
                            fontsize=9,
                        )

            plt.tight_layout()
            plt.show()
        # end of update function

    category_selector.observe(update, names="value")
    metric_selector.observe(update, names="value")
    log_scale.observe(update, names="value")

    display(
        widgets.VBox([
            widgets.HBox([
                category_selector,
                widgets.VBox([
                    metric_selector,
                    log_scale,
                ]),
            ]),
            output,
        ])
    )
    
    update()




            




match MODE:
    case Mode.CNN:
        model = FlowMatching().to(device)
    case Mode.BasicTransformer:
        model = FlowMatching_with_Transformer().to(device)

initial_state = {
    name: param.detach().cpu().clone()
    for name, param in model.named_parameters()
}

# loader = create_loader(training_size=10, batch_size=10)
# train(model, loader)
# show_all_digits(model)
#
# save_model(model)

model = load_model(model)
# show_all_digits(model)

visualize_params(model.named_parameters(), initial_state_dict=initial_state)




In [13]:
for name, param in model.named_parameters():
    print(name, param.shape)

pos_embedding torch.Size([1, 49, 64])
digit_embedding.weight torch.Size([10, 64])
time_embedding.0.weight torch.Size([64, 1])
time_embedding.0.bias torch.Size([64])
time_embedding.2.weight torch.Size([64, 64])
time_embedding.2.bias torch.Size([64])
patch_embedding.weight torch.Size([64, 1, 4, 4])
patch_embedding.bias torch.Size([64])
layers.0.self_attn.in_proj_weight torch.Size([192, 64])
layers.0.self_attn.in_proj_bias torch.Size([192])
layers.0.self_attn.out_proj.weight torch.Size([64, 64])
layers.0.self_attn.out_proj.bias torch.Size([64])
layers.0.linear1.weight torch.Size([128, 64])
layers.0.linear1.bias torch.Size([128])
layers.0.linear2.weight torch.Size([64, 128])
layers.0.linear2.bias torch.Size([64])
layers.0.norm1.weight torch.Size([64])
layers.0.norm1.bias torch.Size([64])
layers.0.norm2.weight torch.Size([64])
layers.0.norm2.bias torch.Size([64])
layers.1.self_attn.in_proj_weight torch.Size([192, 64])
layers.1.self_attn.in_proj_bias torch.Size([192])
layers.1.self_attn.out_

In [68]:
for name, param in model.layers[0].named_parameters():
    if param.requires_grad:
        print(name, param.data)
    

self_attn.in_proj_weight tensor([[ 0.1350, -0.0402,  0.0686,  ...,  0.0389, -0.1454,  0.0406],
        [-0.0068,  0.0761,  0.1572,  ..., -0.0129,  0.0814, -0.1435],
        [ 0.1170, -0.0266,  0.0249,  ..., -0.0588, -0.0368,  0.0791],
        ...,
        [ 0.0201, -0.0417, -0.0757,  ..., -0.1258,  0.1497, -0.1234],
        [ 0.0648,  0.1401,  0.0706,  ..., -0.1224, -0.1064,  0.1337],
        [-0.0750,  0.1126,  0.1362,  ...,  0.1147,  0.1320, -0.0912]])
self_attn.in_proj_bias tensor([-8.1116e-03, -9.6579e-03,  2.3216e-03,  1.1037e-02,  8.3503e-03,
         2.3833e-03,  7.8167e-03,  8.6353e-03, -1.9712e-03,  6.8757e-03,
         1.9769e-03,  8.8905e-03, -3.4706e-03,  8.5316e-03,  4.9175e-03,
        -1.0734e-02,  2.0560e-03,  8.2883e-03, -5.8238e-03, -9.3940e-03,
        -5.3206e-03,  2.6231e-03,  6.0420e-03,  8.5663e-03, -7.4792e-03,
        -7.9534e-03,  1.0210e-02, -9.7438e-03,  6.8258e-03,  2.8100e-03,
        -4.4102e-03,  8.5198e-03,  1.2368e-02,  6.4435e-03,  1.2528e-02,
       

In [10]:
model.named_parameters()

<generator object Module.named_parameters at 0x133f67e40>